In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:

# ============================================================
# MELANOMA SKIN CANCER - COMPLETE EDA
# Supports CSV AND IMAGE FOLDER DATASETS
# ============================================================

import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from PIL import Image
from pathlib import Path

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

# -------------------- 1. DATASET PATH --------------------

DATA_PATH = "/kaggle/input"

print("=" * 60)
print("MELANOMA SKIN CANCER DATASET - EDA")
print("=" * 60)

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        "Kaggle input folder does not exist."
    )

# -------------------- 2. FIND CSV FILES --------------------

csv_files = []

image_extensions = (
    ".jpg", ".jpeg", ".png", ".bmp", ".webp"
)

image_paths = []

for root, dirs, files in os.walk(DATA_PATH):

    for file in files:

        full_path = os.path.join(root, file)

        if file.lower().endswith(".csv"):
            csv_files.append(full_path)

        elif file.lower().endswith(image_extensions):
            image_paths.append(full_path)

print("\nCSV Files Found:", len(csv_files))
print("Images Found:", len(image_paths))

# -------------------- 3. LOAD CSV OR IMAGE DATA --------------------

df = pd.DataFrame()

if csv_files:

    file_path = max(csv_files, key=os.path.getsize)

    print("\nLoading CSV:", file_path)

    df = pd.read_csv(file_path, low_memory=False)

    print("CSV Loaded Successfully!")

else:

    print("\nNo CSV found!")
    print("Creating dataset directly from image folders...")

    image_data = []

    for img_path in image_paths:

        folder_name = Path(img_path).parent.name

        image_data.append({
            "image_name": os.path.basename(img_path),
            "image_path": img_path,
            "label": folder_name
        })

    if image_data:
        df = pd.DataFrame(image_data)

    else:
        raise FileNotFoundError(
            "No CSV or image files found! "
            "Please add your Melanoma dataset using "
            "Kaggle Add Input."
        )

# -------------------- 4. DATASET OVERVIEW --------------------

print("\n1. DATASET OVERVIEW")

print("Total Rows:", df.shape[0])
print("Total Columns:", df.shape[1])

print("\nColumn Names:")
print(df.columns.tolist())

print("\nFirst 5 Records:")
display(df.head())

print("\nDataset Information:")
df.info()

print("\nStatistical Summary:")
display(df.describe(include="all").T)

# -------------------- 5. MISSING VALUES --------------------

print("\n2. MISSING VALUES ANALYSIS")

missing = df.isnull().sum()

missing_df = pd.DataFrame({
    "Missing Values": missing,
    "Percentage": (missing / len(df)) * 100
})

display(missing_df)

missing_df = missing_df[missing_df["Missing Values"] > 0]

if not missing_df.empty:

    plt.figure(figsize=(12, 5))

    sns.barplot(
        x=missing_df.index,
        y=missing_df["Percentage"]
    )

    plt.title("Missing Values Analysis")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

else:
    print("No Missing Values Found!")

# -------------------- 6. DUPLICATE ANALYSIS --------------------

print("\n3. DUPLICATE ANALYSIS")

duplicates = df.duplicated().sum()

print("Total Duplicate Rows:", duplicates)

# -------------------- 7. TARGET COLUMN --------------------

print("\n4. TARGET CLASS ANALYSIS")

target_candidates = [
    "label",
    "target",
    "diagnosis",
    "class",
    "benign_malignant"
]

target_col = next(
    (c for c in target_candidates if c in df.columns),
    None
)

if target_col:

    print("Target Column:", target_col)

    print("\nClass Distribution:")
    print(df[target_col].value_counts())

    plt.figure(figsize=(9, 5))

    sns.countplot(
        data=df,
        x=target_col,
        order=df[target_col].value_counts().index
    )

    plt.title("Melanoma Class Distribution")
    plt.xlabel("Class")
    plt.ylabel("Number of Images")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(7, 7))

    df[target_col].value_counts().plot(
        kind="pie",
        autopct="%1.1f%%",
        startangle=90
    )

    plt.title("Class Percentage")
    plt.ylabel("")
    plt.show()

else:
    print("No target column detected.")

# -------------------- 8. CATEGORICAL ANALYSIS --------------------

print("\n5. CATEGORICAL FEATURES ANALYSIS")

categorical_cols = df.select_dtypes(
    include=["object", "category"]
).columns.tolist()

print("Categorical Columns:", categorical_cols)

for col in categorical_cols[:6]:

    plt.figure(figsize=(9, 4))

    df[col].value_counts().head(10).plot(kind="bar")

    plt.title(f"Distribution of {col}")
    plt.xlabel(col)
    plt.ylabel("Count")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

# -------------------- 9. NUMERICAL ANALYSIS --------------------

print("\n6. NUMERICAL FEATURES ANALYSIS")

numeric_cols = df.select_dtypes(
    include=np.number
).columns.tolist()

print("Numerical Columns:", numeric_cols)

if numeric_cols:

    display(df[numeric_cols].describe().T)

    for col in numeric_cols[:8]:

        plt.figure(figsize=(8, 4))

        sns.histplot(
            df[col].dropna(),
            kde=True,
            bins=30
        )

        plt.title(f"Distribution of {col}")
        plt.tight_layout()
        plt.show()

        plt.figure(figsize=(8, 3))

        sns.boxplot(x=df[col])

        plt.title(f"Boxplot of {col}")
        plt.tight_layout()
        plt.show()

else:
    print("No numerical columns available.")

# -------------------- 10. CORRELATION HEATMAP --------------------

print("\n7. CORRELATION ANALYSIS")

if len(numeric_cols) >= 2:

    plt.figure(figsize=(12, 8))

    sns.heatmap(
        df[numeric_cols].corr(),
        annot=True,
        cmap="coolwarm",
        fmt=".2f"
    )

    plt.title("Feature Correlation Heatmap")
    plt.tight_layout()
    plt.show()

else:
    print("Not enough numerical columns.")

# -------------------- 11. IMAGE VISUALIZATION --------------------

print("\n8. SKIN LESION IMAGE ANALYSIS")

print("Total Images:", len(image_paths))

if image_paths:

    sample_count = min(12, len(image_paths))

    plt.figure(figsize=(15, 10))

    for i, img_path in enumerate(image_paths[:sample_count]):

        try:

            img = Image.open(img_path).convert("RGB")

            plt.subplot(3, 4, i + 1)
            plt.imshow(img)

            plt.title(Path(img_path).parent.name)
            plt.axis("off")

        except Exception:
            continue

    plt.suptitle("Sample Melanoma Skin Images", fontsize=16)
    plt.tight_layout()
    plt.show()

else:
    print("No images available.")

# -------------------- 12. IMAGE DIMENSION ANALYSIS --------------------

print("\n9. IMAGE DIMENSION ANALYSIS")

widths = []
heights = []

for img_path in image_paths[:1000]:

    try:

        img = Image.open(img_path)

        width, height = img.size

        widths.append(width)
        heights.append(height)

    except Exception:
        continue

if widths:

    image_stats = pd.DataFrame({
        "Width": widths,
        "Height": heights
    })

    print("\nImage Statistics:")
    display(image_stats.describe())

    plt.figure(figsize=(8, 5))

    sns.scatterplot(
        data=image_stats,
        x="Width",
        y="Height"
    )

    plt.title("Image Width vs Height")
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(8, 4))

    sns.histplot(widths, bins=20, kde=True)

    plt.title("Image Width Distribution")
    plt.show()

    plt.figure(figsize=(8, 4))

    sns.histplot(heights, bins=20, kde=True)

    plt.title("Image Height Distribution")
    plt.show()

else:
    print("Image dimensions unavailable.")

# -------------------- 13. FINAL EDA SUMMARY --------------------

print("\n" + "=" * 60)
print("FINAL EDA REPORT")
print("=" * 60)

print("Total Records:", len(df))
print("Total Features:", len(df.columns))
print("Total Missing Values:", df.isnull().sum().sum())
print("Duplicate Records:", duplicates)
print("Numerical Features:", len(numeric_cols))
print("Categorical Features:", len(categorical_cols))
print("Total Images:", len(image_paths))

if target_col:
    print("\nTarget Distribution:")
    print(df[target_col].value_counts())

print("\nEDA COMPLETED SUCCESSFULLY!")
print("=" * 60)


In [ ]:

# ============================================================
# MELANOMA SKIN CANCER CLASSIFICATION USING CNN
# COMPLETE DEEP LEARNING PROJECT
# ============================================================

import os
import warnings
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from pathlib import Path
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    accuracy_score
)

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

# -------------------- 1. CONFIGURATION --------------------

DATA_PATH = "/kaggle/input"

IMAGE_SIZE = (128, 128)
BATCH_SIZE = 32
EPOCHS = 20
SEED = 42

print("=" * 60)
print("MELANOMA SKIN CANCER - CNN CLASSIFICATION")
print("=" * 60)

print("TensorFlow Version:", tf.__version__)
print("GPU Available:", tf.config.list_physical_devices("GPU"))

# -------------------- 2. FIND DATASET FOLDER --------------------

image_extensions = {
    ".jpg", ".jpeg", ".png", ".bmp", ".webp"
}

candidates = []

for root, dirs, files in os.walk(DATA_PATH):

    class_folders = []
    total_images = 0

    for folder in dirs:

        folder_path = os.path.join(root, folder)

        try:
            count = sum(
                1 for f in os.listdir(folder_path)
                if Path(f).suffix.lower() in image_extensions
            )
        except PermissionError:
            count = 0

        if count > 0:
            class_folders.append(folder)
            total_images += count

    if len(class_folders) >= 2:

        score = total_images

        if "train" in root.lower():
            score *= 1.2

        candidates.append(
            (score, root, class_folders, total_images)
        )

if not candidates:
    raise FileNotFoundError(
        "No class folders found! Dataset must contain "
        "at least two folders with labeled images."
    )

candidates.sort(reverse=True)

_, dataset_dir, detected_classes, total_images = candidates[0]

print("\nDataset Folder:", dataset_dir)
print("Detected Classes:", detected_classes)
print("Total Images:", total_images)

# -------------------- 3. LOAD DATASET --------------------

train_ds = tf.keras.utils.image_dataset_from_directory(
    dataset_dir,
    validation_split=0.20,
    subset="training",
    seed=SEED,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="int"
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    dataset_dir,
    validation_split=0.20,
    subset="validation",
    seed=SEED,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="int",
    shuffle=False
)

class_names = train_ds.class_names
num_classes = len(class_names)

print("\nClass Names:", class_names)
print("Number of Classes:", num_classes)

# -------------------- 4. DISPLAY SAMPLE IMAGES --------------------

plt.figure(figsize=(12, 8))

for images, labels in train_ds.take(1):

    for i in range(min(9, len(images))):

        plt.subplot(3, 3, i + 1)

        plt.imshow(images[i].numpy().astype("uint8"))

        plt.title(class_names[labels[i].numpy()])
        plt.axis("off")

plt.suptitle("Melanoma Dataset Sample Images")
plt.tight_layout()
plt.show()

# -------------------- 5. DATA PREPROCESSING --------------------

AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)

# Calculate class weights for imbalanced datasets

train_labels = np.concatenate([
    labels.numpy() for _, labels in train_ds
])

classes = np.unique(train_labels)

weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=train_labels
)

class_weights = {
    int(c): float(w)
    for c, w in zip(classes, weights)
}

print("\nClass Weights:", class_weights)

# -------------------- 6. BUILD CNN MODEL --------------------

data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1)
])

model = models.Sequential([

    layers.Input(shape=(128, 128, 3)),

    data_augmentation,

    layers.Rescaling(1./255),

    # First CNN Block
    layers.Conv2D(32, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),

    # Second CNN Block
    layers.Conv2D(64, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),

    # Third CNN Block
    layers.Conv2D(128, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),

    # Fourth CNN Block
    layers.Conv2D(256, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),

    # Classification Layers
    layers.Flatten(),

    layers.Dense(128, activation="relu"),

    layers.Dropout(0.5),

    layers.Dense(num_classes, activation="softmax")
])

# -------------------- 7. MODEL SUMMARY --------------------

print("\nCNN MODEL SUMMARY")

model.summary()

# -------------------- 8. COMPILE MODEL --------------------

model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# -------------------- 9. CALLBACKS --------------------

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=2,
    min_lr=0.00001
)

# -------------------- 10. TRAIN CNN MODEL --------------------

print("\nSTARTING CNN TRAINING...")

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    class_weight=class_weights,
    callbacks=[early_stop, reduce_lr]
)

print("\nMODEL TRAINING COMPLETED!")

# -------------------- 11. MODEL EVALUATION --------------------

print("\nMODEL EVALUATION")

loss, accuracy = model.evaluate(val_ds)

print("Validation Loss:", round(loss, 4))
print("Validation Accuracy:", round(accuracy * 100, 2), "%")

# -------------------- 12. ACCURACY GRAPH --------------------

plt.figure(figsize=(10, 5))

plt.plot(
    history.history["accuracy"],
    label="Training Accuracy"
)

plt.plot(
    history.history["val_accuracy"],
    label="Validation Accuracy"
)

plt.title("CNN Model Accuracy")
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.legend()
plt.show()

# -------------------- 13. LOSS GRAPH --------------------

plt.figure(figsize=(10, 5))

plt.plot(
    history.history["loss"],
    label="Training Loss"
)

plt.plot(
    history.history["val_loss"],
    label="Validation Loss"
)

plt.title("CNN Model Loss")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.legend()
plt.show()

# -------------------- 14. PREDICTIONS --------------------

y_true = np.concatenate([
    labels.numpy() for _, labels in val_ds
])

predictions = model.predict(val_ds)

y_pred = np.argmax(predictions, axis=1)

print("\nPrediction Accuracy:",
      round(accuracy_score(y_true, y_pred) * 100, 2), "%")

# -------------------- 15. CONFUSION MATRIX --------------------

cm = confusion_matrix(
    y_true,
    y_pred,
    labels=np.arange(num_classes)
)

plt.figure(figsize=(8, 6))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names
)

plt.title("CNN Confusion Matrix")
plt.xlabel("Predicted Class")
plt.ylabel("Actual Class")
plt.show()

# -------------------- 16. CLASSIFICATION REPORT --------------------

print("\nCLASSIFICATION REPORT")

print(
    classification_report(
        y_true,
        y_pred,
        labels=np.arange(num_classes),
        target_names=class_names,
        zero_division=0
    )
)

# -------------------- 17. SAVE MODEL --------------------

model.save("melanoma_cnn_model.keras")

print("\nModel Saved Successfully!")
print("File: melanoma_cnn_model.keras")

# -------------------- 18. FINAL SUMMARY --------------------

print("\n" + "=" * 60)
print("FINAL CNN RESULTS")
print("=" * 60)

print("Total Images:", total_images)
print("Number of Classes:", num_classes)
print("Classes:", class_names)
print("Epochs Completed:", len(history.history["loss"]))
print("Validation Accuracy:", round(accuracy * 100, 2), "%")
print("Validation Loss:", round(loss, 4))

print("\nCNN CLASSIFICATION COMPLETED!")
print("=" * 60)


In [ ]:
# Save the current CNN model
model.save("/kaggle/working/melanoma_cnn_model.keras")

print("Current CNN model saved successfully!")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

sns.set_theme(style="whitegrid")

# Find target column
target_col = next(
    (c for c in ["label", "target", "diagnosis",
                 "class", "benign_malignant"]
     if c in df.columns),
    None
)

if target_col:
    plt.figure(figsize=(9, 5))
    sns.countplot(data=df, x=target_col)
    plt.title("Melanoma Class Distribution")
    plt.xlabel("Skin Cancer Classes")
    plt.ylabel("Number of Images")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
else:
    print("Target column not found.")

In [ ]:
if target_col:
    plt.figure(figsize=(8, 8))

    df[target_col].value_counts().plot(
        kind="pie",
        autopct="%1.1f%%",
        startangle=90
    )

    plt.title("Melanoma Skin Cancer Class Percentage")
    plt.ylabel("")
    plt.show()

In [ ]:
import matplotlib.pyplot as plt

loss, accuracy = model.evaluate(val_ds, verbose=0)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Accuracy Graph
axes[0].bar(["Accuracy"], [accuracy * 100])
axes[0].set_ylim(0, 100)
axes[0].set_title("CNN Accuracy")
axes[0].set_ylabel("Percentage (%)")

# Loss Graph
axes[1].bar(["Loss"], [loss])
axes[1].set_title("CNN Loss")
axes[1].set_ylabel("Loss Value")

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

# Evaluate existing CNN model
loss, accuracy = model.evaluate(val_ds, verbose=0)

# CNN Loss Graph
plt.figure(figsize=(9, 5))

plt.bar(
    ["Validation Loss"],
    [loss]
)

plt.title("CNN Model Loss")
plt.xlabel("Model")
plt.ylabel("Loss Value")

plt.text(
    0,
    loss,
    f"{loss:.4f}",
    ha="center",
    va="bottom"
)

plt.tight_layout()
plt.show()

print("CNN Validation Loss:", round(loss, 4))

In [ ]:
from sklearn.metrics import confusion_matrix

y_true = np.concatenate([
    labels.numpy() for images, labels in val_ds
])

predictions = model.predict(val_ds)

y_pred = np.argmax(predictions, axis=1)

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(8, 6))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names
)

plt.title("CNN Confusion Matrix")
plt.xlabel("Predicted Class")
plt.ylabel("Actual Class")
plt.show()

In [ ]:
from sklearn.metrics import classification_report

print("CNN Classification Report")

print(classification_report(
    y_true,
    y_pred,
    target_names=class_names,
    zero_division=0
))

In [ ]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_true, y_pred) * 100

performance = [
    accuracy,
    100 - accuracy
]

plt.figure(figsize=(7, 7))

plt.pie(
    performance,
    labels=["Correct Predictions", "Incorrect Predictions"],
    autopct="%1.2f%%",
    startangle=90
)

plt.title("CNN Model Prediction Performance")
plt.show()

In [ ]:
plt.figure(figsize=(12, 8))

for images, labels in val_ds.take(1):

    predictions = model.predict(images)

    for i in range(min(9, len(images))):

        plt.subplot(3, 3, i + 1)

        plt.imshow(images[i].numpy().astype("uint8"))

        actual = class_names[labels[i].numpy()]
        predicted = class_names[np.argmax(predictions[i])]

        plt.title(
            f"Actual: {actual}\nPredicted: {predicted}"
        )

        plt.axis("off")

plt.suptitle("Actual vs Predicted Skin Images")
plt.tight_layout()
plt.show()

In [ ]:
print("=" * 45)
print("MELANOMA CNN MODEL PERFORMANCE")
print("=" * 45)

print("Accuracy:", round(accuracy, 2), "%")

print("Total Validation Images:", len(y_true))

print("Correct Predictions:", np.sum(y_true == y_pred))

print("Incorrect Predictions:", np.sum(y_true != y_pred))

print("Number of Classes:", len(class_names))

print("=" * 45)